# InternVL3 Hybrid Processor - Full Batch Evaluation - FIXED VERSION

**COMPREHENSIVE TESTING**: Process all images in evaluation_data and compare against ground truth.

**Target**: Achieve 82% accuracy restoration using the hybrid architecture.

**Architecture**: 
- InternVL3 model for accuracy
- Llama's proven processing pipeline for reliability
- ExtractionCleaner for value normalization
- Document-aware field selection

**Evaluation Goals**:
- Process all images in `/home/jovyan/nfs_share/tod/evaluation_data/`
- Compare against `/home/jovyan/nfs_share/tod/evaluation_data/ground_truth.csv`
- Generate comprehensive accuracy metrics
- Benchmark against 82% target baseline

In [ ]:
# Import hybrid processor and Llama analytics infrastructure
try:
    from models.document_aware_internvl3_hybrid_processor import DocumentAwareInternVL3HybridProcessor
    from common.extraction_parser import discover_images, parse_extraction_response
    from common.evaluation_metrics import calculate_field_accuracy, load_ground_truth
    
    # Import Llama analytics infrastructure to avoid code duplication
    from common.batch_analytics import BatchAnalytics
    from common.batch_reporting import BatchReporter
    from common.batch_visualizations import BatchVisualizer
    
    print("✅ InternVL3 Hybrid Processor imported successfully")
    print("✅ Evaluation utilities imported successfully")
    print("✅ Llama analytics infrastructure imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Check that the project root path is correct")
    raise

In [ ]:
# Configuration following Llama infrastructure pattern
import os
import glob
import json
import time
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from rich import print as rprint
from rich.console import Console

# Environment-specific base paths (following Llama pattern)
ENVIRONMENT_BASES = {
    'sandbox': '/home/jovyan/nfs_share/tod',
    'efs': '/efs/shared/PoC_data'
}
base_data_path = ENVIRONMENT_BASES['sandbox']

CONFIG = {
    # Model settings
    'MODEL_PATH': '/home/jovyan/nfs_share/models/InternVL3-8B',
    
    # Batch settings - Using base path for consistency with Llama
    'DATA_DIR': f'{base_data_path}/evaluation_data',
    'GROUND_TRUTH': f'{base_data_path}/evaluation_data/ground_truth.csv',
    'OUTPUT_BASE': f'{base_data_path}/output',
    'MAX_IMAGES': None,  # None for all, or set limit
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Verbosity control
    'VERBOSE': True,
    'SHOW_PROMPTS': True,
    
    # InternVL3 optimization settings
    'USE_QUANTIZATION': False,
    'DEVICE_MAP': 'auto', 
    'MAX_NEW_TOKENS': 600,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True
}

# InternVL3 prompt configuration
PROMPT_CONFIG = {
    'detection_file': 'prompts/document_type_detection.yaml',
    'detection_key': 'detection_internvl3',  # Use optimized InternVL3 detection
    'extraction_files': {
        'INVOICE': 'prompts/internvl3_prompts.yaml',
        'RECEIPT': 'prompts/internvl3_prompts.yaml', 
        'BANK_STATEMENT': 'prompts/internvl3_prompts.yaml'
    },
    'extraction_keys': {
        'INVOICE': 'invoice',
        'RECEIPT': 'receipt',
        'BANK_STATEMENT': 'bank_statement'
    }
}

console = Console()
print("✅ Configuration set up following Llama infrastructure pattern")
print(f"📂 Evaluation data: {CONFIG['DATA_DIR']}")
print(f"📊 Ground truth: {CONFIG['GROUND_TRUTH']}")
print(f"🤖 Model path: {CONFIG['MODEL_PATH']}")
print(f"📁 Output base: {CONFIG['OUTPUT_BASE']}")

In [ ]:
# Output Directory Setup - Following Llama infrastructure pattern

# Convert OUTPUT_BASE to Path and handle absolute/relative paths
OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
if not OUTPUT_BASE.is_absolute():
    # If relative, make it relative to current working directory
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

for dir_path in OUTPUT_DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

# Image Discovery using Llama pattern
data_dir = Path(CONFIG['DATA_DIR'])
if not data_dir.is_absolute():
    data_dir = Path.cwd() / data_dir

ground_truth_path = Path(CONFIG['GROUND_TRUTH'])
if not ground_truth_path.is_absolute():
    ground_truth_path = Path.cwd() / ground_truth_path

# Discover images from the resolved data directory
all_images = discover_images(str(data_dir))

# Load ground truth from the resolved path
ground_truth = load_ground_truth(str(ground_truth_path), verbose=CONFIG['VERBOSE'])

# Apply filters
if CONFIG['DOCUMENT_TYPES']:
    filtered = []
    for img in all_images:
        img_name = Path(img).name
        if img_name in ground_truth:
            doc_type = ground_truth[img_name].get('DOCUMENT_TYPE', '').lower()
            if any(dt.lower() in doc_type for dt in CONFIG['DOCUMENT_TYPES']):
                filtered.append(img)
    all_images = filtered

if CONFIG['MAX_IMAGES']:
    all_images = all_images[:CONFIG['MAX_IMAGES']]

rprint(f"[bold green]📁 Output directories created[/bold green]")
rprint(f"[cyan]📂 Base output: {OUTPUT_BASE}[/cyan]")
rprint(f"[bold green]Ready to process {len(all_images)} images[/bold green]")
rprint(f"[cyan]Data directory: {data_dir}[/cyan]")
rprint(f"[cyan]Ground truth: {ground_truth_path}[/cyan]")
for i, img in enumerate(all_images[:5], 1):
    print(f"  {i}. {Path(img).name}")
if len(all_images) > 5:
    print(f"  ... and {len(all_images) - 5} more")

In [4]:
# Load ground truth data
print("📊 Loading ground truth data...")

try:
    ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH)
    print(f"✅ Ground truth loaded: {len(ground_truth_df)} records")
    print(f"📋 Ground truth columns: {list(ground_truth_df.columns)}")
    
    # Show sample of ground truth data
    print("\n📄 Sample ground truth data:")
    print(ground_truth_df.head(3))
    
    # Check for required columns
    required_columns = ['image_file', 'DOCUMENT_TYPE']
    missing_columns = [col for col in required_columns if col not in ground_truth_df.columns]
    if missing_columns:
        print(f"⚠️ Missing required columns: {missing_columns}")
    else:
        print("✅ All required columns present")
        
except FileNotFoundError:
    print(f"❌ Ground truth file not found: {GROUND_TRUTH_PATH}")
    raise
except Exception as e:
    print(f"❌ Error loading ground truth: {e}")
    raise

📊 Loading ground truth data...
✅ Ground truth loaded: 9 records
📋 Ground truth columns: ['image_file', 'DOCUMENT_TYPE', 'BUSINESS_ABN', 'BUSINESS_ADDRESS', 'GST_AMOUNT', 'INVOICE_DATE', 'IS_GST_INCLUDED', 'LINE_ITEM_DESCRIPTIONS', 'LINE_ITEM_QUANTITIES', 'LINE_ITEM_PRICES', 'LINE_ITEM_TOTAL_PRICES', 'PAYER_ADDRESS', 'PAYER_NAME', 'STATEMENT_DATE_RANGE', 'SUPPLIER_NAME', 'TOTAL_AMOUNT', 'TRANSACTION_AMOUNTS_PAID', 'TRANSACTION_DATES', 'TRANSACTION_AMOUNTS_RECEIVED', 'ACCOUNT_BALANCE']

📄 Sample ground truth data:
      image_file DOCUMENT_TYPE    BUSINESS_ABN  \
0  image_001.png       RECEIPT  06 082 698 025   
1  image_002.png       RECEIPT  29 466 483 258   
2  image_004.png       RECEIPT  66 658 925 499   

                    BUSINESS_ADDRESS GST_AMOUNT INVOICE_DATE IS_GST_INCLUDED  \
0    481 Bourke Street Perth WA 6000      $8.62   05/08/2025            true   
1  680 Collins Street Darwin NT 0800      $5.20   18/07/2025            true   
2     993 Pitt Street Darwin NT 0800     

In [5]:
# Define field sets for document-aware processing
INVOICE_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT"
]

RECEIPT_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT"
]

BANK_STATEMENT_FIELDS = [
    "DOCUMENT_TYPE", "STATEMENT_DATE_RANGE", "LINE_ITEM_DESCRIPTIONS",
    "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID", "TRANSACTION_AMOUNTS_RECEIVED",
    "ACCOUNT_BALANCE"
]

UNIVERSAL_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "STATEMENT_DATE_RANGE",
    "LINE_ITEM_DESCRIPTIONS", "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES",
    "LINE_ITEM_TOTAL_PRICES", "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT",
    "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID", "TRANSACTION_AMOUNTS_RECEIVED",
    "ACCOUNT_BALANCE"
]

print(f"📋 Invoice fields: {len(INVOICE_FIELDS)}")
print(f"📋 Receipt fields: {len(RECEIPT_FIELDS)}")
print(f"📋 Bank statement fields: {len(BANK_STATEMENT_FIELDS)}")
print(f"📋 Universal fields: {len(UNIVERSAL_FIELDS)}")
print("\n✅ Field sets defined for document-aware processing")

📋 Invoice fields: 14
📋 Receipt fields: 14
📋 Bank statement fields: 7
📋 Universal fields: 19

✅ Field sets defined for document-aware processing


In [ ]:
# Initialize InternVL3 Hybrid Processor with BatchDocumentProcessor
print("🚀 Initializing InternVL3 Hybrid Processor with Two-Step Architecture...")

try:
    # Step 1: Initialize the hybrid processor
    hybrid_processor = DocumentAwareInternVL3HybridProcessor(
        field_list=UNIVERSAL_FIELDS,
        model_path=INTERNVL3_MODEL_PATH,
        debug=False  # Disable debug for batch processing
    )

    print("✅ HYBRID PROCESSOR INITIALIZED SUCCESSFULLY")
    print(f"📊 Field count: {hybrid_processor.field_count}")
    print(f"🎯 Model path: {hybrid_processor.model_path}")
    print(f"🧹 ExtractionCleaner: {'✅ Active' if hybrid_processor.cleaner else '❌ Missing'}")
    print(f"🔧 Device: {hybrid_processor.device}")
    print(f"🚀 Model loaded: {'✅ Yes' if hybrid_processor.model else '❌ No'}")

    # Get model info
    model_info = hybrid_processor.get_model_info()
    print(f"📋 Model info: {model_info}")

    # Step 2: Configure for BatchDocumentProcessor with two-step workflow
    PROMPT_CONFIG = {
        'detection_file': 'prompts/document_type_detection.yaml',
        'detection_key': 'detection_internvl3',  # 🔧 FIXED: Use optimized InternVL3 detection
        'extraction_files': {
            'INVOICE': 'prompts/internvl3_prompts.yaml',       # Use optimized InternVL3 prompts
            'RECEIPT': 'prompts/internvl3_prompts.yaml',       # Use optimized InternVL3 prompts
            'BANK_STATEMENT': 'prompts/internvl3_prompts.yaml' # Use optimized InternVL3 prompts
        },
        'extraction_keys': {
            'INVOICE': 'invoice',
            'RECEIPT': 'receipt',
            'BANK_STATEMENT': 'bank_statement'
        }
    }

    # Step 3: Initialize BatchDocumentProcessor with InternVL3 hybrid processor
    from common.batch_processor import BatchDocumentProcessor
    from rich.console import Console

    console = Console()

    # Pass the hybrid processor as the model parameter (InternVL3 handler)
    batch_processor = BatchDocumentProcessor(
        model=hybrid_processor,  # Hybrid processor acts as InternVL3 handler
        processor=None,          # Not needed for InternVL3
        prompt_config=PROMPT_CONFIG,
        ground_truth_csv=GROUND_TRUTH_PATH,
        console=console
    )

    print("✅ BATCH PROCESSOR INITIALIZED WITH OPTIMIZED TWO-STEP ARCHITECTURE")
    print("🔍 Step 1: Document type detection using 'detection_internvl3' prompt")
    print("📊 Step 2: Document-specific extraction using optimized prompts")
    print("🎯 Architecture: Optimized Detection → Classification → Enhanced Extraction")

except Exception as e:
    print(f"❌ Processor initialization failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [7]:
# Batch processing function
def process_image_batch(image_files, processor, batch_name="Batch"):
    """Process a batch of images and return results."""
    print(f"\n🔄 Processing {batch_name}: {len(image_files)} images")
    print("=" * 60)
    
    results = []
    processing_times = []
    
    for i, image_path in enumerate(image_files, 1):
        image_name = os.path.basename(image_path)
        print(f"\n📷 Processing {i}/{len(image_files)}: {image_name}")
        
        try:
            start_time = time.time()
            
            # Process single image
            result = processor.process_single_image(image_path)
            
            processing_time = time.time() - start_time
            processing_times.append(processing_time)
            
            # Add metadata to result
            result['image_file'] = image_name
            result['image_path'] = image_path
            result['processing_time'] = processing_time
            result['timestamp'] = datetime.now().isoformat()
            
            results.append(result)
            
            # Progress update
            fields_extracted = result['extracted_fields_count']
            total_fields = result['field_count']
            completeness = result['response_completeness']
            
            print(f"  ✅ Fields: {fields_extracted}/{total_fields} ({completeness:.1%}) | Time: {processing_time:.2f}s")
            
        except Exception as e:
            print(f"  ❌ Error processing {image_name}: {e}")
            
            # Create error result
            error_result = {
                'image_file': image_name,
                'image_path': image_path,
                'error': str(e),
                'extracted_fields_count': 0,
                'field_count': len(UNIVERSAL_FIELDS),
                'response_completeness': 0.0,
                'processing_time': 0.0,
                'timestamp': datetime.now().isoformat(),
                'extracted_data': {field: 'ERROR' for field in UNIVERSAL_FIELDS}
            }
            results.append(error_result)
    
    # Batch summary
    avg_time = sum(processing_times) / len(processing_times) if processing_times else 0
    total_time = sum(processing_times)
    successful_extractions = sum(1 for r in results if r.get('extracted_fields_count', 0) > 0)
    
    print(f"\n📊 {batch_name} Summary:")
    print(f"  Images processed: {len(results)}")
    print(f"  Successful extractions: {successful_extractions}/{len(results)}")
    print(f"  Average processing time: {avg_time:.2f}s")
    print(f"  Total processing time: {total_time:.2f}s")
    
    return results

print("✅ Batch processing function defined")

✅ Batch processing function defined


In [ ]:
# Process all images with working InternVL3 logic - KEEP WORKING APPROACH
print("🚀 STARTING INTERNVL3 BATCH PROCESSING")
print("=" * 80)
print("🔍 Architecture: Document Detection → Document-Specific Extraction")  
print("📝 Detection Prompt: prompts/document_type_detection.yaml")
print("📊 Extraction Prompts: prompts/internvl3_prompts.yaml")
print("=" * 80)

# Process images using working InternVL3 approach
batch_results = []
processing_times = []
document_types_found = {}

for i, image_path in enumerate(all_images, 1):
    image_name = os.path.basename(image_path)
    print(f"\n📷 Processing {i}/{len(all_images)}: {image_name}")
    
    try:
        start_time = time.time()
        
        # Use the working hybrid processor (avoid BatchDocumentProcessor recursion issues)
        result = hybrid_processor.process_single_image(image_path)
        
        processing_time = time.time() - start_time
        processing_times.append(processing_time)
        
        # Extract document type for analytics
        document_type = result.get('extracted_data', {}).get('DOCUMENT_TYPE', 'UNKNOWN')
        document_types_found[document_type] = document_types_found.get(document_type, 0) + 1
        
        # Format result for analytics compatibility (matching Llama structure)
        formatted_result = {
            'image_name': image_name,
            'image_path': image_path,
            'document_type': document_type.lower() if document_type != 'UNKNOWN' else 'unknown',
            'processing_time': processing_time,
            'timestamp': datetime.now().isoformat(),
            'extraction_result': {
                'extracted_data': result.get('extracted_data', {}),
                'field_count': result.get('field_count', 0),
                'extracted_fields_count': result.get('extracted_fields_count', 0),
                'response_completeness': result.get('response_completeness', 0.0)
            }
        }
        
        # Add evaluation against ground truth (for analytics)
        if image_name in ground_truth:
            gt_data = ground_truth[image_name]
            
            # Calculate field matches for analytics
            extracted_data = result.get('extracted_data', {})
            fields_matched = 0
            fields_extracted = 0
            total_fields = len(extracted_data)
            
            for field, extracted_value in extracted_data.items():
                if extracted_value != 'NOT_FOUND':
                    fields_extracted += 1
                    if field in gt_data and gt_data[field] != 'NOT_FOUND':
                        # Simple exact match for now
                        if str(extracted_value).lower().strip() == str(gt_data[field]).lower().strip():
                            fields_matched += 1
            
            overall_accuracy = fields_matched / total_fields if total_fields > 0 else 0
            
            formatted_result['evaluation'] = {
                'overall_accuracy': overall_accuracy,
                'fields_extracted': fields_extracted,
                'fields_matched': fields_matched,
                'total_fields': total_fields
            }
        
        batch_results.append(formatted_result)
        
        # Progress update
        fields_extracted = result['extracted_fields_count']
        total_fields = result['field_count']
        completeness = result['response_completeness']
        
        print(f"  ✅ Fields: {fields_extracted}/{total_fields} ({completeness:.1%}) | Time: {processing_time:.2f}s")
        
    except Exception as e:
        print(f"  ❌ Error processing {image_name}: {e}")
        processing_time = time.time() - start_time
        processing_times.append(processing_time)
        
        # Create error result compatible with analytics
        error_result = {
            'image_name': image_name,
            'image_path': image_path,
            'document_type': 'error',
            'processing_time': processing_time,
            'timestamp': datetime.now().isoformat(),
            'extraction_result': {
                'extracted_data': {},
                'field_count': 0,
                'extracted_fields_count': 0,
                'response_completeness': 0.0
            },
            'error': str(e)
        }
        batch_results.append(error_result)

print(f"\n🎉 BATCH PROCESSING COMPLETE")
print(f"📊 Total results: {len(batch_results)}")
print(f"⏱️ Processing times: {len(processing_times)}")
print(f"📋 Document types found: {document_types_found}")
print(f"📅 Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Brief summary
rprint(f"[bold green]✅ Processed {len(batch_results)} images[/bold green]")
rprint(f"[cyan]Average time: {np.mean(processing_times):.2f}s[/cyan]")
rprint(f"[cyan]Document types: {list(document_types_found.keys())}[/cyan]")

In [ ]:
# Generate Analytics using Llama infrastructure - NO CODE DUPLICATION
rprint("\n[bold blue]📊 Generating Comprehensive Analytics[/bold blue]")

# Create analytics using proven Llama infrastructure
analytics = BatchAnalytics(batch_results, processing_times)

# Generate and save DataFrames using established patterns
saved_files, df_results, df_summary, df_doctype_stats, df_field_stats = analytics.save_all_dataframes(
    OUTPUT_DIRS['csv'], BATCH_TIMESTAMP, verbose=CONFIG['VERBOSE']
)

# Display key results
rprint("\n[bold blue]📊 InternVL3 Results Summary[/bold blue]")
display(df_summary)

# Create model-specific CSV matching Llama pattern for model comparison
FIELD_COLUMNS = [
    'DOCUMENT_TYPE', 'BUSINESS_ABN', 'SUPPLIER_NAME', 'BUSINESS_ADDRESS', 
    'PAYER_NAME', 'PAYER_ADDRESS', 'INVOICE_DATE', 'LINE_ITEM_DESCRIPTIONS',
    'LINE_ITEM_QUANTITIES', 'LINE_ITEM_PRICES', 'LINE_ITEM_TOTAL_PRICES',
    'IS_GST_INCLUDED', 'GST_AMOUNT', 'TOTAL_AMOUNT', 'STATEMENT_DATE_RANGE',
    'TRANSACTION_DATES', 'TRANSACTION_AMOUNTS_PAID', 'TRANSACTION_AMOUNTS_RECEIVED',
    'ACCOUNT_BALANCE'
]

# Create comprehensive results data matching Llama structure for model comparison
internvl3_csv_data = []

for i, result in enumerate(batch_results):
    # Basic metadata
    image_name = result['image_name']
    doc_type = result.get('document_type', 'unknown')
    processing_time = result['processing_time']
    
    # Extract fields from result
    extraction_result = result.get('extraction_result', {})
    extracted_fields = extraction_result.get('extracted_data', {})
    evaluation_data = result.get('evaluation', {})
    
    # Count fields
    total_fields = len(FIELD_COLUMNS)
    found_fields = sum(1 for field in FIELD_COLUMNS if extracted_fields.get(field, 'NOT_FOUND') != 'NOT_FOUND')
    field_coverage = (found_fields / total_fields * 100) if total_fields > 0 else 0
    
    # Calculate accuracy
    overall_accuracy = evaluation_data.get('overall_accuracy', 0) * 100 if evaluation_data else 0  # Convert to %
    fields_extracted = evaluation_data.get('fields_extracted', 0) if evaluation_data else 0
    fields_matched = evaluation_data.get('fields_matched', 0) if evaluation_data else 0
    eval_total_fields = evaluation_data.get('total_fields', total_fields) if evaluation_data else total_fields
    
    # Create prompt identifier
    prompt_used = f"internvl3_{doc_type}" if doc_type != 'unknown' else "internvl3_unknown"
    
    # Create row data
    row_data = {
        'image_file': image_name,
        'image_name': image_name,
        'document_type': doc_type,
        'processing_time': processing_time,
        'field_count': eval_total_fields,
        'found_fields': fields_extracted,
        'field_coverage': (fields_extracted / eval_total_fields * 100) if eval_total_fields > 0 else 0,
        'prompt_used': prompt_used,
        'timestamp': result['timestamp'],
        'overall_accuracy': overall_accuracy,
        'fields_extracted': fields_extracted,
        'fields_matched': fields_matched,
        'total_fields': eval_total_fields
    }
    
    # Add all field values
    for field in FIELD_COLUMNS:
        row_data[field] = extracted_fields.get(field, 'NOT_FOUND')
    
    internvl3_csv_data.append(row_data)

# Create DataFrame and save
internvl3_df = pd.DataFrame(internvl3_csv_data)
internvl3_csv_path = OUTPUT_DIRS['csv'] / f"internvl3_batch_results_{BATCH_TIMESTAMP}.csv"
internvl3_df.to_csv(internvl3_csv_path, index=False)

rprint("[bold green]✅ InternVL3 model-specific CSV exported:[/bold green]")
rprint(f"[cyan]📄 File: {internvl3_csv_path}[/cyan]")
rprint(f"[cyan]📊 Structure: {len(internvl3_df)} rows × {len(internvl3_df.columns)} columns[/cyan]")
rprint("[cyan]🔗 Compatible with model_comparison.ipynb pattern: *internvl3*batch*results*.csv[/cyan]")

# Display sample of the exported data
rprint("\n[bold blue]📋 Sample exported data (first 3 rows, key columns):[/bold blue]")
sample_cols = ['image_file', 'document_type', 'overall_accuracy', 'processing_time', 'found_fields', 'field_coverage']
if len(internvl3_df) > 0:
    display(internvl3_df[sample_cols].head(3))
    
    # Show accuracy distribution
    accuracy_stats = internvl3_df['overall_accuracy'].describe()
    rprint(f"\n[bold blue]📈 Accuracy Statistics:[/bold blue]")
    rprint(f"[cyan]Mean: {accuracy_stats['mean']:.1f}%[/cyan]")
    rprint(f"[cyan]Median: {accuracy_stats['50%']:.1f}%[/cyan]")
    rprint(f"[cyan]Min: {accuracy_stats['min']:.1f}%[/cyan]")
    rprint(f"[cyan]Max: {accuracy_stats['max']:.1f}%[/cyan]")
else:
    rprint("[yellow]⚠️ No data to display[/yellow]")

In [ ]:
# Create Visualizations using Llama infrastructure - NO CODE DUPLICATION
rprint("\n[bold blue]📊 Creating Visualizations and Dashboard[/bold blue]")

# Create visualizations using proven Llama infrastructure
visualizer = BatchVisualizer()

viz_files = visualizer.create_all_visualizations(
    df_results, 
    df_doctype_stats,
    OUTPUT_DIRS['visualizations'],
    BATCH_TIMESTAMP,
    show=False  # Disable inline display to reduce output, but files are saved
)

rprint(f"[bold green]✅ Visualizations created and saved[/bold green]")
for viz_file in viz_files:
    rprint(f"[cyan]📊 {viz_file}[/cyan]")

# Display dashboard if available  
dashboard_files = list(OUTPUT_DIRS['visualizations'].glob(f"dashboard_{BATCH_TIMESTAMP}.png"))
if dashboard_files:
    from IPython.display import Image, display
    dashboard_path = dashboard_files[0]
    rprint(f"\n[bold blue]📊 InternVL3 Visual Dashboard:[/bold blue]")
    rprint(f"[cyan]📄 File: {dashboard_path}[/cyan]")
    # Optionally display the dashboard inline (uncomment if desired)
    # display(Image(str(dashboard_path)))
else:
    rprint(f"\n[yellow]⚠️ Dashboard not found in {OUTPUT_DIRS['visualizations']}[/yellow]")

In [ ]:
# Generate Reports using Llama infrastructure - NO CODE DUPLICATION
rprint("\n[bold blue]📄 Generating Comprehensive Reports[/bold blue]")

# Generate reports using proven Llama infrastructure
reporter = BatchReporter(
    batch_results, 
    processing_times,
    document_types_found,
    BATCH_TIMESTAMP
)

# Save all reports using CONFIG verbose setting
report_files = reporter.save_all_reports(
    OUTPUT_DIRS,
    df_results,
    df_summary,
    df_doctype_stats,
    CONFIG['MODEL_PATH'],
    {
        'data_dir': CONFIG['DATA_DIR'],
        'ground_truth': CONFIG['GROUND_TRUTH'],
        'max_images': CONFIG['MAX_IMAGES'],
        'document_types': CONFIG['DOCUMENT_TYPES']
    },
    {
        'use_quantization': CONFIG['USE_QUANTIZATION'],
        'device_map': CONFIG['DEVICE_MAP'],
        'max_new_tokens': CONFIG['MAX_NEW_TOKENS'],
        'torch_dtype': CONFIG['TORCH_DTYPE'],
        'low_cpu_mem_usage': CONFIG['LOW_CPU_MEM_USAGE']
    },
    verbose=CONFIG['VERBOSE']
)

rprint(f"[bold green]✅ Reports generated and saved[/bold green]")
for report_file in report_files:
    rprint(f"[cyan]📄 {report_file}[/cyan]")

In [ ]:
# Display Final Summary - Following Llama Infrastructure Pattern
console.rule("[bold green]InternVL3 Batch Processing Complete[/bold green]")

total_images = len(batch_results)
successful = len([r for r in batch_results if 'error' not in r])
avg_accuracy = df_results['overall_accuracy'].mean() if len(df_results) > 0 else 0
avg_processing_time = np.mean(processing_times) if processing_times else 0

rprint(f"[bold green]✅ Processed: {total_images} images[/bold green]")
rprint(f"[cyan]Success Rate: {(successful/total_images*100):.1f}%[/cyan]")
rprint(f"[cyan]Average Accuracy: {avg_accuracy:.2f}%[/cyan]")
rprint(f"[cyan]Average Processing Time: {avg_processing_time:.2f}s[/cyan]")
rprint(f"[cyan]Output Base: {OUTPUT_BASE}[/cyan]")

# Performance comparison with target
target_accuracy = 82.0
if avg_accuracy >= target_accuracy:
    rprint(f"[bold green]🎉 TARGET ACHIEVED: {avg_accuracy:.1f}% ≥ {target_accuracy}%[/bold green]")
else:
    gap = target_accuracy - avg_accuracy
    rprint(f"[yellow]📈 TARGET GAP: {gap:.1f}% improvement needed to reach {target_accuracy}%[/yellow]")

# Document type distribution
rprint(f"\n[bold blue]📋 Document Type Distribution:[/bold blue]")
for doc_type, count in document_types_found.items():
    percentage = (count / total_images * 100) if total_images > 0 else 0
    rprint(f"[cyan]  {doc_type}: {count} documents ({percentage:.1f}%)[/cyan]")

# Show key output files
rprint(f"\n[bold blue]📁 Key Output Files:[/bold blue]")
rprint(f"[cyan]📊 Results CSV: {internvl3_csv_path.name}[/cyan]")
rprint(f"[cyan]📈 Analytics: {len(saved_files)} CSV files generated[/cyan]")
rprint(f"[cyan]📊 Visualizations: {len(viz_files)} charts created[/cyan]")
rprint(f"[cyan]📄 Reports: {len(report_files)} reports generated[/cyan]")

# Architecture summary
rprint(f"\n[bold blue]🏗️ Architecture Summary:[/bold blue]")
rprint(f"[cyan]✅ InternVL3 Hybrid Processor: Working (no recursion issues)[/cyan]")
rprint(f"[cyan]✅ Llama Analytics Infrastructure: Integrated[/cyan]")
rprint(f"[cyan]✅ Code Duplication: Eliminated[/cyan]")
rprint(f"[cyan]✅ Model Comparison: Compatible CSV format[/cyan]")

rprint(f"\n[bold green]🎯 InternVL3 batch processing complete with full Llama infrastructure integration![/bold green]")

In [13]:
# Save results to files - FIXED VERSION
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = '/home/jovyan/nfs_share/tod/output'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Saving results to {output_dir}...")

# Save detailed results
results_file = f"{output_dir}/internvl3_hybrid_batch_results_FIXED_{timestamp}.csv"
results_df.to_csv(results_file, index=False)
print(f"✅ Detailed results saved: {results_file}")

# Save evaluation data if available
if len(evaluation_df) > 0:
    evaluation_file = f"{output_dir}/internvl3_hybrid_evaluation_FIXED_{timestamp}.csv"
    evaluation_df.to_csv(evaluation_file, index=False)
    print(f"✅ Evaluation data saved: {evaluation_file}")
    
    # Save accuracy summary
    accuracy_file = f"{output_dir}/internvl3_hybrid_accuracy_FIXED_{timestamp}.csv"
    accuracy_df.to_csv(accuracy_file, index=False)
    print(f"✅ Accuracy summary saved: {accuracy_file}")

# Function to convert numpy types to native Python types
def convert_numpy_types(obj):
    """Recursively convert numpy types in dict to native Python types."""
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    else:
        return obj

# Save summary statistics - FIXED: Use avg_coverage instead of avg_completeness
summary_stats = {
    'timestamp': timestamp,
    'total_images_processed': int(len(results_df)),
    'successful_extractions': int(successful_extractions),
    'success_rate': float(successful_extractions / len(results_df)) if len(results_df) > 0 else 0.0,
    'average_field_coverage': float(avg_coverage) if not pd.isna(avg_coverage) else 0.0,  # FIXED: This was the issue!
    'average_processing_time': float(avg_processing_time) if not pd.isna(avg_processing_time) else 0.0,
    'overall_accuracy': float(overall_accuracy) if len(evaluation_df) > 0 and not pd.isna(overall_accuracy) else None,
    'target_accuracy': 0.82,
    'target_achieved': bool(overall_accuracy >= 0.82) if len(evaluation_df) > 0 else None,
    'model_path': INTERNVL3_MODEL_PATH,
    'field_count': len(UNIVERSAL_FIELDS),
    'architecture': 'InternVL3 Hybrid (InternVL3 + Llama Pipeline) - FIXED VERSION'
}

# Convert the entire dict to ensure all numpy types are handled
summary_stats = convert_numpy_types(summary_stats)

summary_file = f"{output_dir}/internvl3_hybrid_summary_FIXED_{timestamp}.json"
try:
    with open(summary_file, 'w') as f:
        json.dump(summary_stats, f, indent=2)
    print(f"✅ Summary statistics saved: {summary_file}")
except Exception as e:
    print(f"⚠️ Error saving summary: {e}")
    # Save as string representation if JSON fails
    summary_file_txt = f"{output_dir}/internvl3_hybrid_summary_FIXED_{timestamp}.txt"
    with open(summary_file_txt, 'w') as f:
        f.write(str(summary_stats))
    print(f"📝 Summary saved as text: {summary_file_txt}")

print(f"\n🎉 EVALUATION COMPLETE - FIXED VERSION!")
print(f"📂 All results saved to: {output_dir}")
print(f"📅 Timestamp: {timestamp}")
print(f"🔧 FIXED: No more 'avg_completeness' variable errors!")

💾 Saving results to /home/jovyan/nfs_share/tod/output...
✅ Detailed results saved: /home/jovyan/nfs_share/tod/output/internvl3_hybrid_batch_results_FIXED_20250919_080353.csv
✅ Evaluation data saved: /home/jovyan/nfs_share/tod/output/internvl3_hybrid_evaluation_FIXED_20250919_080353.csv
✅ Accuracy summary saved: /home/jovyan/nfs_share/tod/output/internvl3_hybrid_accuracy_FIXED_20250919_080353.csv
✅ Summary statistics saved: /home/jovyan/nfs_share/tod/output/internvl3_hybrid_summary_FIXED_20250919_080353.json

🎉 EVALUATION COMPLETE - FIXED VERSION!
📂 All results saved to: /home/jovyan/nfs_share/tod/output
📅 Timestamp: 20250919_080353
🔧 FIXED: No more 'avg_completeness' variable errors!


In [14]:
# Final Summary Report - FIXED VERSION
print("🎯 INTERNVL3 HYBRID PROCESSOR - FINAL EVALUATION REPORT - FIXED VERSION")
print("=" * 80)

print(f"\n📊 PROCESSING STATISTICS:")
print(f"  Total images processed: {len(results_df)}")
print(f"  Successful extractions: {successful_extractions}/{len(results_df)} ({successful_extractions/len(results_df):.1%})")
print(f"  Average field coverage: {avg_coverage:.1f}%")  # FIXED: Use avg_coverage
print(f"  Average processing time: {avg_processing_time:.2f}s per image")

if len(evaluation_df) > 0:
    print(f"\n🎯 ACCURACY ASSESSMENT:")
    print(f"  Overall accuracy: {overall_accuracy:.1%}")
    print(f"  Target accuracy: 82.0%")
    
    if overall_accuracy >= 0.82:
        print(f"  🎉 STATUS: TARGET ACHIEVED (+{(overall_accuracy - 0.82)*100:.1f}pp above target)")
    else:
        gap = 0.82 - overall_accuracy
        print(f"  📈 STATUS: {gap*100:.1f}pp improvement needed to reach target")
    
    # Top performing fields
    top_fields = accuracy_df.head(5)
    print(f"\n🏆 TOP PERFORMING FIELDS:")
    for _, row in top_fields.iterrows():
        print(f"  ✅ {row['field']:<25} {row['accuracy']:.1%}")
    
    # Fields needing improvement
    low_fields = accuracy_df[accuracy_df['accuracy'] < 0.6]
    if len(low_fields) > 0:
        print(f"\n⚠️ FIELDS NEEDING IMPROVEMENT (<60% accuracy):")
        for _, row in low_fields.iterrows():
            print(f"  ❌ {row['field']:<25} {row['accuracy']:.1%}")

print(f"\n🏗️ ARCHITECTURE PERFORMANCE:")
print(f"  ✅ InternVL3 model integration: Working")
print(f"  ✅ Llama processing pipeline: Active")
print(f"  ✅ ExtractionCleaner: Integrated")
print(f"  ✅ Document-aware processing: {len(UNIVERSAL_FIELDS)} fields")
print(f"  ✅ Variable error fixes: Complete")

print(f"\n🚀 NEXT STEPS:")
if len(evaluation_df) > 0 and overall_accuracy >= 0.82:
    print(f"  1. 🎉 Hybrid processor ready for production deployment")
    print(f"  2. 📊 Consider A/B testing against current InternVL3 handler")
    print(f"  3. 🔄 Plan migration strategy from existing systems")
    print(f"  4. 📈 Monitor performance in production environment")
elif len(evaluation_df) > 0:
    print(f"  1. 🔧 Focus optimization on underperforming fields")
    print(f"  2. 📝 Review prompt engineering for specific document types")
    print(f"  3. 🧹 Enhance ExtractionCleaner rules for problematic fields")
    print(f"  4. 🔄 Iterate on field-specific processing logic")
else:
    print(f"  1. 🔍 Investigate ground truth data alignment issues")
    print(f"  2. 📊 Verify image file naming consistency")
    print(f"  3. 🔄 Re-run evaluation with corrected data matching")

print(f"\n💾 OUTPUT FILES (FIXED VERSION):")
print(f"  📄 Detailed results: internvl3_hybrid_batch_results_FIXED_{timestamp}.csv")
if len(evaluation_df) > 0:
    print(f"  📊 Evaluation data: internvl3_hybrid_evaluation_FIXED_{timestamp}.csv")
    print(f"  🎯 Accuracy summary: internvl3_hybrid_accuracy_FIXED_{timestamp}.csv")
print(f"  📋 Summary statistics: internvl3_hybrid_summary_FIXED_{timestamp}.json")

print(f"\n" + "=" * 80)
print(f"🎯 INTERNVL3 HYBRID EVALUATION COMPLETE - NO MORE VARIABLE ERRORS!")
print(f"🔧 FIXED: All 'avg_completeness' references replaced with 'avg_coverage'")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"" + "=" * 80)

🎯 INTERNVL3 HYBRID PROCESSOR - FINAL EVALUATION REPORT - FIXED VERSION

📊 PROCESSING STATISTICS:
  Total images processed: 9
  Successful extractions: 9/9 (100.0%)
  Average field coverage: 77.8%
  Average processing time: 9.38s per image

🎯 ACCURACY ASSESSMENT:
  Overall accuracy: 37.1%
  Target accuracy: 82.0%
  📈 STATUS: 44.9pp improvement needed to reach target

🏆 TOP PERFORMING FIELDS:
  ✅ STATEMENT_DATE_RANGE      100.0%
  ✅ INVOICE_DATE              66.7%
  ✅ SUPPLIER_NAME             66.7%
  ✅ GST_AMOUNT                66.7%
  ✅ TOTAL_AMOUNT              66.7%

⚠️ FIELDS NEEDING IMPROVEMENT (<60% accuracy):
  ❌ PAYER_NAME                55.6%
  ❌ TRANSACTION_AMOUNTS_RECEIVED 33.3%
  ❌ BUSINESS_ABN              33.3%
  ❌ LINE_ITEM_QUANTITIES      33.3%
  ❌ IS_GST_INCLUDED           33.3%
  ❌ TRANSACTION_DATES         33.3%
  ❌ LINE_ITEM_DESCRIPTIONS    22.2%
  ❌ BUSINESS_ADDRESS          16.7%
  ❌ DOCUMENT_TYPE             11.1%
  ❌ LINE_ITEM_PRICES          0.0%
  ❌ LINE_ITEM_T